## **Inspeção dos dados brutos — ENEM 2025**

- **Objetivo** : Inspecionar o arquivo de participantes do ENEM 2025 para identificar os passos de limpeza necessários
- **Processo** : Carregamento da base, seguido da avaliação de estrutura, valores ausentes, unicidade e conformidade com o dicionário oficial

In [1]:
## Carregamento das bibliotecas e criação dos caminhos de diretórios

import os
from pathlib import Path
import pandas as pd
import numpy as np
from dotenv import load_dotenv
pd.options.display.max_columns = None
pd.options.display.float_format = "{:.3f}".format

load_dotenv()

DATA_DIR = Path(os.environ["ENEM_RAW_DIR"]) / "DADOS"
PARTICIPANTS_FILE = DATA_DIR / "PARTICIPANTES_2025.csv"

In [2]:
## carregamento da base de participantes

participantes = pd.read_csv(PARTICIPANTS_FILE, sep=";", encoding="latin-1")

In [3]:
# Dicionários pra validação

## Variáveis do participante
dominios_participantes = {
    "NU_ANO": {2025},
    "TP_FAIXA_ETARIA": set(range(1, 21)),
    "TP_SEXO": {"M", "F"},
    "TP_ESTADO_CIVIL": set(range(0, 5)),
    "TP_COR_RACA": set(range(0, 6)),
    "TP_NACIONALIDADE": set(range(0, 5)),
    "TP_ST_CONCLUSAO": set(range(1, 5)),
    "TP_ANO_CONCLUIU": set(range(0, 20)),
    "TP_ENSINO": {1, 2},
    "IN_TREINEIRO": {0, 1},
}

## Códigos dos Estados

codigo_uf = {
    11: "RO",
    12: "AC",
    13: "AM",
    14: "RR",
    15: "PA",
    16: "AP",
    17: "TO",
    21: "MA",
    22: "PI",
    23: "CE",
    24: "RN",
    25: "PB",
    26: "PE",
    27: "AL",
    28: "SE",
    29: "BA",
    31: "MG",
    32: "ES",
    33: "RJ",
    35: "SP",
    41: "PR",
    42: "SC",
    43: "RS",
    50: "MS",
    51: "MT",
    52: "GO",
    53: "DF",
}

## Questionário Socioeconômico
dominios_questionario = {
    "Q001": set("ABCDEFGH"),
    "Q002": set("ABCDEFGH"),
    "Q003": set("ABCDEF"),
    "Q004": set("ABCDEF"),
    "Q005": set(range(1, 21)),
    "Q006": set("AB"),
    "Q007": set("ABCDEFGHIJKLMNOPQ"),
    "Q008": set("ABCD"),
    "Q009": set("ABCD"),
    "Q010": set("ABCD"),
    "Q011": set("ABCD"),
    "Q012": set("ABCD"),
    "Q013": set("ABCD"),
    "Q014": set("AB"),
    "Q015": set("AB"),
    "Q016": set("AB"),
    "Q017": set("AB"),
    "Q018": set("ABCD"),
    "Q019": set("AB"),
    "Q020": set("AB"),
    "Q021": set("ABCDE"),
    "Q022": set("ABCDE"),
    "Q023": set("ABCDEF"),
}



In [4]:
## visualização inicial dos dados e dos tipos de dados
display (participantes.head(10))
participantes.info()

,NU_INSCRICAO,NU_ANO,TP_FAIXA_ETARIA,TP_SEXO,TP_ESTADO_CIVIL,TP_COR_RACA,TP_NACIONALIDADE,TP_ST_CONCLUSAO,TP_ANO_CONCLUIU,TP_ENSINO,IN_TREINEIRO,CO_MUNICIPIO_PROVA,NO_MUNICIPIO_PROVA,CO_UF_PROVA,SG_UF_PROVA,Q001,Q002,Q003,Q004,Q005,Q006,Q007,Q008,Q009,Q010,Q011,Q012,Q013,Q014,Q015,Q016,Q017,Q018,Q019,Q020,Q021,Q022,Q023
0,210066506229,2025,6,F,1,2,2,1,4,NaN,0,2932200,Ubaitaba,29,BA,C,F,A,A,1,B,B,A,B,B,A,A,B,A,A,A,A,A,A,A,A,B,A
1,210066506230,2025,3,M,1,1,1,2,0,1.000,0,2910800,Feira de Santana,29,BA,B,E,C,B,3,A,D,A,B,C,B,A,B,A,B,B,A,C,A,B,B,B,A
2,210066506231,2025,2,F,1,1,1,3,0,NaN,1,3549805,São José do Rio Preto,35,SP,E,E,D,D,6,A,F,A,C,D,A,B,B,A,A,B,A,A,A,B,C,C,C
3,210066506232,2025,2,M,1,3,2,2,0,1.000,0,2203909,Floriano,22,PI,D,F,D,D,3,A,B,A,B,C,A,B,B,A,B,A,A,B,A,B,B,D,D
4,210066506233,2025,1,F,1,1,1,3,0,NaN,1,5101803,Barra do Garças,51,MT,G,G,E,E,4,B,Q,D,D,D,C,A,D,B,B,B,B,D,B,B,D,E,D
5,210066506234,2025,7,M,1,1,1,1,4,NaN,0,4304606,Canoas,43,RS,E,E,C,C,5,B,G,A,B,D,B,A,B,A,B,A,A,B,A,B,B,E,A
6,210066506235,2025,3,M,1,1,1,2,0,1.000,0,4202404,Blumenau,42,SC,D,D,C,B,4,B,G,A,B,C,B,A,B,A,B,B,B,B,B,B,A,D,A
7,210066506236,2025,12,F,1,1,1,1,18,NaN,0,3162807,São João Evangelista,31,MG,C,B,B,B,4,B,G,A,B,C,B,A,B,A,A,B,A,B,A,B,B,D,A
8,210066506237,2025,3,M,1,2,1,2,0,NaN,0,2925105,Poções,29,BA,H,H,B,B,4,B,B,A,B,C,A,A,B,A,A,A,A,B,A,B,A,B,A
9,210066506238,2025,2,F,1,1,1,2,0,1.000,0,2610509,Passira,26,PE,D,E,A,A,5,A,B,A,B,C,A,A,B,A,A,A,A,B,A,B,A,C,A


<class 'pandas.DataFrame'>
RangeIndex: 4810772 entries, 0 to 4810771
Data columns (total 38 columns):
 #   Column              Dtype  
---  ------              -----  
 0   NU_INSCRICAO        int64  
 1   NU_ANO              int64  
 2   TP_FAIXA_ETARIA     int64  
 3   TP_SEXO             str    
 4   TP_ESTADO_CIVIL     int64  
 5   TP_COR_RACA         int64  
 6   TP_NACIONALIDADE    int64  
 7   TP_ST_CONCLUSAO     int64  
 8   TP_ANO_CONCLUIU     int64  
 9   TP_ENSINO           float64
 10  IN_TREINEIRO        int64  
 11  CO_MUNICIPIO_PROVA  int64  
 12  NO_MUNICIPIO_PROVA  str    
 13  CO_UF_PROVA         int64  
 14  SG_UF_PROVA         str    
 15  Q001                str    
 16  Q002                str    
 17  Q003                str    
 18  Q004                str    
 19  Q005                int64  
 20  Q006                str    
 21  Q007                str    
 22  Q008                str    
 23  Q009                str    
 24  Q010                str    
 25  Q0

In [5]:
## Perfil geral dos dados e verificação de valores nulos 
perfil_de_qualidade = {
    "tipo": participantes.dtypes,
    "nulos": participantes.isnull().sum(),
    "percentual_nulos": participantes.isnull().sum() / len(participantes["NU_INSCRICAO"]) * 100,
    "valores_unicos": participantes.nunique(),
}
tabela_perfil_qualidade = pd.DataFrame(perfil_de_qualidade)
tabela_perfil_qualidade

,tipo,nulos,percentual_nulos,valores_unicos
NU_INSCRICAO,int64,0,0.000,4810772
NU_ANO,int64,0,0.000,1
TP_FAIXA_ETARIA,int64,0,0.000,20
TP_SEXO,str,0,0.000,2
TP_ESTADO_CIVIL,int64,0,0.000,5
TP_COR_RACA,int64,0,0.000,6
TP_NACIONALIDADE,int64,0,0.000,5
TP_ST_CONCLUSAO,int64,0,0.000,4
TP_ANO_CONCLUIU,int64,0,0.000,20
TP_ENSINO,float64,3080608,64.036,2


- A base de participantes possui 4810772 linhas com 38 colunas
- Os números de inscrição não possuem valores nulos e possuem um único valor por registro
- A base possui exclusivamente dados de 2025, sem erros de coleta
- TP_ENSINO é a única coluna com valores ausentes, correspondendo a 64,04% dos registros. O padrão será investigado em conjunto com TP_ST_CONCLUSAO.
- Não foi preciso analisar o describe para as métricas numéricas porque os valores são categóricos

In [6]:
 ## Valores esperados para as variáveis dos participantes

resultado_validacao = []

for coluna, valor in dominios_participantes.items():
    resultado_validacao.append({
        "coluna": coluna,
        "valor_esperado": sorted(valor),
        "invalidos": (participantes[coluna].notna() & ~participantes[coluna].isin(valor)).sum()
    })      

validacao_dominios = pd.DataFrame(resultado_validacao)
validacao_dominios

,coluna,valor_esperado,invalidos
0,NU_ANO,[2025],0
1,TP_FAIXA_ETARIA,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",0
2,TP_SEXO,"[F, M]",0
3,TP_ESTADO_CIVIL,"[0, 1, 2, 3, 4]",0
4,TP_COR_RACA,"[0, 1, 2, 3, 4, 5]",0
5,TP_NACIONALIDADE,"[0, 1, 2, 3, 4]",0
6,TP_ST_CONCLUSAO,"[1, 2, 3, 4]",0
7,TP_ANO_CONCLUIU,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",0
8,TP_ENSINO,"[1, 2]",0
9,IN_TREINEIRO,"[0, 1]",0


In [7]:
#Validação de estados aplicados

validacao_estados = (participantes[["CO_UF_PROVA", "SG_UF_PROVA"]].drop_duplicates().sort_values("CO_UF_PROVA", ignore_index=True))
validacao_estados["SG_UF_ESPERADA"] = (validacao_estados["CO_UF_PROVA"].map(codigo_uf))
validacao_estados["invalidos"] = np.where(validacao_estados["SG_UF_PROVA"] != validacao_estados["SG_UF_ESPERADA"],1,0)
quantidade_estados = participantes["SG_UF_PROVA"].nunique()

display(validacao_estados)
print(f"Estados observados: {quantidade_estados}")


,CO_UF_PROVA,SG_UF_PROVA,SG_UF_ESPERADA,invalidos
0,11,RO,RO,0
1,12,AC,AC,0
2,13,AM,AM,0
3,14,RR,RR,0
4,15,PA,PA,0
5,16,AP,AP,0
6,17,TO,TO,0
7,21,MA,MA,0
8,22,PI,PI,0
9,23,CE,CE,0


Estados observados: 27


In [8]:
#Validação das respostas do questionário

resultado_questionario = []

for pergunta,resposta in dominios_questionario.items():

    resultado_questionario.append({
    "pergunta": pergunta,
    "respostas_invalidas" : (participantes[pergunta].notna() & ~participantes[pergunta].isin(resposta)).sum()
    })

validacao_questionario = pd.DataFrame(resultado_questionario)
validacao_questionario

,pergunta,respostas_invalidas
0,Q001,0
1,Q002,0
2,Q003,0
3,Q004,0
4,Q005,0
5,Q006,0
6,Q007,0
7,Q008,0
8,Q009,0
9,Q010,0


 ### **Conclusões Iniciais**

 - As variáveis relacionadas ao perfil do participante, ao local de prova e ao questionário socioeconômico apresentaram valores compatíveis com os domínios definidos no dicionário oficial
 - Não foram encontrados códigos inesperados nas variáveis categóricas nem respostas inválidas entre Q001 e Q023
 - Entre os participantes que declararam já ter concluído o Ensino Médio, 209.996 possuem o código 0, correspondente à categoria oficial “Não informado”. Para preservar as demais informações presentes esses registros serão preservados sem imputação
 - Como a inspeção busca apenas o registro de informação e não conclusões específicas sobre os dados, que justificariam exclusão de registros, remoção de colunas ou imputações de valores, a limpeza irá se concentrar na adequação e padronização dos dados

### **Plano de Limpeza**

-  Registrar as métricas iniciais e as conclusões

- Adequar os tipos de dados conforme o significado de identificadores, códigos e variáveis quantitativas.

- Padronizar as colunas textuais

- Preservar ausências e categorias oficiais conforme documentado anteriormente

- Validar o resultado, comparando estrutura, valores ausentes, unicidade e domínios antes e depois da limpeza.

- Salvar a base limpa em Parquet